# NB17a: Earnings Call Error Mining

**Part 1 of 2**: Train baseline → identify earnings call errors → export CSV for VS augmentation.

**Workflow**:
1. **NB17a** (Kaggle GPU): Train baseline, evaluate, mine errors, export `earnings_errors_for_augmentation.csv`
2. **Local** (Claude Code + `verbalized-sampling-augment` skill): Generate augmented data
3. **NB17b** (Kaggle GPU): Train with extended context + augmented data + curriculum + class weights

## 0. Setup

In [ ]:
!pip install -q peft datasets transformers accelerate

In [ ]:
import torchimport numpy as npimport pandas as pdfrom datasets import load_dataset, Dataset, concatenate_datasetsfrom transformers import (    AutoTokenizer, AutoModelForSequenceClassification,    TrainingArguments, Trainer, training_args,)from peft import LoraConfig, get_peft_model, TaskTypefrom sklearn.metrics import accuracy_score, f1_score, classification_reportfrom tqdm import tqdmimport json, gc, reimport matplotlib.pyplot as pltimport seaborn as snsNUM_CLASSES = 3LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]MODEL_NAME = "answerdotai/ModernBERT-base"SEED = 3407FPB_SOURCE = 5SOURCE_NAMES = {3: "Earnings (Narrative)", 4: "Press Releases", 5: "FPB", 8: "Earnings (Q&A)", 9: "Tweets"}torch.manual_seed(SEED); np.random.seed(SEED)device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device: {device}")if device.type == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading & Truncation Analysis

In [ ]:
label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}raw_ds = load_dataset("neoyipeng/financial_reasoning_aggregated")raw_ds = raw_ds.filter(lambda x: x["task"] == "sentiment")print(f"Train: {len(raw_ds['train']):,}  Val: {len(raw_ds['validation']):,}  Test: {len(raw_ds['test']):,}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)all_train = concatenate_datasets([raw_ds["train"], raw_ds["validation"]])texts_all = all_train["text"]; sources_all = np.array(all_train["source"])token_counts = np.array([len(tokenizer(t, truncation=False, return_attention_mask=False)["input_ids"]) for t in tqdm(texts_all, desc="Tokenizing")])print(f"\n{'Source':<25} {'N':>6} {'Median':>7} {'P95':>7} {'Trunc@512':>10} {'Trunc@1024':>11}")print("-" * 75)for src in sorted(set(sources_all)):    mask = sources_all == src; tc = token_counts[mask]; name = SOURCE_NAMES.get(src, f"Src {src}")    t512 = (tc > 512).sum(); t1024 = (tc > 1024).sum()    print(f"{name:<25} {mask.sum():>6} {np.median(tc):>7.0f} {np.percentile(tc, 95):>7.0f} {t512:>5} ({t512/mask.sum()*100:>4.1f}%) {t1024:>5} ({t1024/mask.sum()*100:>4.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))for src in sorted(set(sources_all)):    mask = sources_all == src; name = SOURCE_NAMES.get(src, f"Src {src}")    axes[0].hist(token_counts[mask], bins=50, alpha=0.5, label=f"{name} (n={mask.sum()})")axes[0].axvline(512, color="red", ls="--", lw=2, label="512 (current)")axes[0].axvline(1024, color="orange", ls="--", lw=2, label="1024 (proposed)")axes[0].set_xlabel("Tokens"); axes[0].set_ylabel("Freq"); axes[0].set_title("Token Length by Source")axes[0].legend(fontsize=7); axes[0].set_xlim(0, 2000)s = np.sort(token_counts); c = np.arange(1, len(s)+1)/len(s)axes[1].plot(s, c, color="steelblue", lw=2)axes[1].axvline(512, color="red", ls="--", label=f"512: {(token_counts<=512).mean():.1%}")axes[1].axvline(1024, color="orange", ls="--", label=f"1024: {(token_counts<=1024).mean():.1%}")axes[1].set_xlabel("Tokens"); axes[1].set_ylabel("Cumulative"); axes[1].set_title("Coverage"); axes[1].legend()plt.tight_layout(); plt.savefig("nb17a_truncation.png", dpi=150, bbox_inches="tight"); plt.show()

## 2. Prepare Datasets

In [ ]:
def prep(dataset):    return dataset.map(lambda ex: {"labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]].tolist()},        remove_columns=[c for c in dataset.column_names if c not in ("text", "labels", "source")])all_tv = concatenate_datasets([raw_ds["train"], raw_ds["validation"]])train_ds = prep(all_tv.filter(lambda x: x["source"] != FPB_SOURCE))test_ds = prep(raw_ds["test"].filter(lambda x: x["source"] != FPB_SOURCE))fpb_50 = load_dataset("takala/financial_phrasebank", "sentences_50agree", split="train", trust_remote_code=True)fpb_all = load_dataset("takala/financial_phrasebank", "sentences_allagree", split="train", trust_remote_code=True)print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}  FPB50: {len(fpb_50):,}  FPB-all: {len(fpb_all):,}")

## 3. Load Existing DataBoost

In [ ]:
import base64, gzip, ioVS_DATA_B64 = "..."vs_raw = gzip.decompress(base64.b64decode(VS_DATA_B64))vs_df = pd.read_csv(io.BytesIO(vs_raw))aug_ds = Dataset.from_dict({"text": vs_df["text"].tolist(), "labels": [np.eye(NUM_CLASSES)[l].tolist() for l in vs_df["label"]]})print(f"Existing DataBoost: {len(aug_ds):,}")

## 4. Training & Evaluation Functions

In [ ]:
def eval_fpb(model, tok, fpb, ml=512, bs=32):    txts, lbls, preds = fpb["sentence"], fpb["label"], []    model.eval()    with torch.no_grad():        for i in range(0, len(txts), bs):            inp = tok(txts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=ml)            inp = {k: v.to(device) for k, v in inp.items()}            preds.extend(torch.argmax(model(**inp).logits, dim=-1).cpu().numpy())    yt, yp = np.array(lbls), np.array(preds)    return {"accuracy": round(accuracy_score(yt, yp), 4), "macro_f1": round(f1_score(yt, yp, average="macro"), 4)}def eval_src(model, tok, ds, ml=512, bs=32):    txts, li, srcs, preds = ds["text"], np.argmax(ds["labels"], axis=1), np.array(ds["source"]), []    model.eval()    with torch.no_grad():        for i in range(0, len(txts), bs):            inp = tok(txts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=ml)            inp = {k: v.to(device) for k, v in inp.items()}            preds.extend(torch.argmax(model(**inp).logits, dim=-1).cpu().numpy())    yt, yp = li, np.array(preds)    r = {"overall_acc": round(accuracy_score(yt, yp), 4)}    print(f"\n  {'Source':<25} {'N':>5} {'Acc':>8}")    print(f"  {'-'*40}")    for s in sorted(set(srcs)):        m = srcs == s        if m.sum() < 3: continue        a = accuracy_score(yt[m], yp[m])        print(f"  {SOURCE_NAMES.get(s, f'Src {s}'):<25} {m.sum():>5} {a:>8.4f}")        r[f"source_{s}_acc"] = round(a, 4)    ec = np.isin(srcs, [3, 8])    if ec.sum():        a = accuracy_score(yt[ec], yp[ec]); r["earnings_acc"] = round(a, 4)        print(f"  {'EARNINGS (3+8)':<25} {ec.sum():>5} {a:>8.4f}")    return rdef full_eval(model, tok, ml=512, label=""):    print(f"\n{'='*60}\n{label}\n{'='*60}")    r50 = eval_fpb(model, tok, fpb_50, ml); print(f"FPB50: {r50['accuracy']:.4f} acc, {r50['macro_f1']:.4f} F1")    ra = eval_fpb(model, tok, fpb_all, ml); print(f"FPB-all: {ra['accuracy']:.4f} acc")    rs = eval_src(model, tok, test_ds, ml)    return {"fpb_50_acc": r50["accuracy"], "fpb_50_f1": r50["macro_f1"], "fpb_all_acc": ra["accuracy"], **rs}

In [ ]:
def train_model(train_dataset, max_length=512, output_dir="out", epochs=10, lr=2e-4, label=""):    print(f"\n{'='*60}\nTRAINING: {label}\n  max_length={max_length}, n={len(train_dataset):,}\n{'='*60}")    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)    tok = AutoTokenizer.from_pretrained(MODEL_NAME)    lora = LoraConfig(r=16, lora_alpha=32, target_modules=["Wqkv","out_proj","Wi","Wo"], lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_CLS)    model = get_peft_model(model, lora); model.gradient_checkpointing_enable(); model = model.to(device)    cols = [c for c in train_dataset.column_names if c not in ("text","labels")]    td = train_dataset.map(lambda ex: tok(ex["text"], truncation=True, max_length=max_length), batched=True, remove_columns=cols)    sp = td.train_test_split(test_size=0.1, seed=SEED)    bs = 8 if max_length <= 512 else 4; ga = 4 if max_length <= 512 else 8    trainer = Trainer(model=model, processing_class=tok, train_dataset=sp["train"], eval_dataset=sp["test"],        args=TrainingArguments(output_dir=output_dir, per_device_train_batch_size=bs, gradient_accumulation_steps=ga,            warmup_steps=10, fp16=True, optim=training_args.OptimizerNames.ADAMW_TORCH, learning_rate=lr,            weight_decay=0.001, lr_scheduler_type="cosine", seed=SEED, num_train_epochs=epochs,            load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,            save_strategy="epoch", eval_strategy="epoch", logging_strategy="epoch",            gradient_checkpointing=True, report_to="none", save_total_limit=1),        compute_metrics=lambda ep: {"accuracy": accuracy_score(ep[1].argmax(axis=-1), ep[0].argmax(axis=-1))})    trainer.train(); model = model.to(device).eval()    return model, tok

## 5. Train Baseline & Evaluate

In [ ]:
baseline_train = concatenate_datasets([train_ds, aug_ds]).shuffle(seed=SEED)baseline_model, baseline_tok = train_model(baseline_train, 512, "out_17a_base", label="BASELINE (512, LoRA+DB)")baseline_results = full_eval(baseline_model, baseline_tok, 512, "BASELINE")with open("nb17a_baseline.json", "w") as f: json.dump(baseline_results, f, indent=2)

## 6. Intervention A: Extended Context (1024)

In [ ]:
model_a, tok_a = train_model(baseline_train, 1024, "out_17a_ext", label="A: Extended (1024)")results_a = full_eval(model_a, tok_a, 1024, "A: Extended Context")with open("nb17a_extended.json", "w") as f: json.dump(results_a, f, indent=2)del model_a; gc.collect(); torch.cuda.empty_cache()

## 7. Mine Earnings Call Errors

In [ ]:
ec_data = all_tv.filter(lambda x: x["source"] in [3, 8])ec_texts, ec_labels = ec_data["text"], np.array([label_dict[l] for l in ec_data["label"]])ec_sources = np.array(ec_data["source"]); ec_preds = []baseline_model.eval()with torch.no_grad():    for i in tqdm(range(0, len(ec_texts), 32), desc="Scanning"):        b = ec_texts[i:i+32]        inp = baseline_tok(b, return_tensors="pt", padding=True, truncation=True, max_length=512)        inp = {k: v.to(device) for k, v in inp.items()}        ec_preds.extend(torch.argmax(baseline_model(**inp).logits, dim=-1).cpu().numpy())ec_preds = np.array(ec_preds); errors = ec_preds != ec_labelsprint(f"Errors: {errors.sum()} / {len(ec_labels)} ({errors.mean():.1%})")hedging_re = re.compile(r'\b(may|might|could|possibly|potentially|likely|unlikely|perhaps|appears?\s+to|seems?\s+to)\b', re.I)conditional_re = re.compile(r'\b(if|although|despite|however|nevertheless|while|whereas|even\s+though)\b', re.I)comparative_re = re.compile(r'\b(more\s+than|less\s+than|higher|lower|increased|decreased|grew|declined|rose|fell|exceeded|missed)\b', re.I)cats = []for t in ec_texts:    c = []    if hedging_re.search(t): c.append("hedging")    if conditional_re.search(t): c.append("conditional")    if comparative_re.search(t): c.append("comparative")    if not c: c.append("other")    cats.append("|".join(c))

## 8. Export Errors CSV

**Download this file** after the notebook runs, then use it with the VS skill in Claude Code.

In [ ]:
rows = []for idx in np.where(errors)[0]:    rows.append({"text": ec_texts[idx], "true_label": int(ec_labels[idx]),        "true_label_name": LABEL_NAMES[ec_labels[idx]], "pred_label": int(ec_preds[idx]),        "pred_label_name": LABEL_NAMES[ec_preds[idx]], "source": int(ec_sources[idx]),        "source_name": SOURCE_NAMES.get(int(ec_sources[idx]),"?"), "linguistic_pattern": cats[idx],        "word_count": len(ec_texts[idx].split())})errors_df = pd.DataFrame(rows)pri = {"hedging": 0, "conditional": 1, "comparative": 2, "other": 3}errors_df["priority"] = errors_df["linguistic_pattern"].apply(lambda x: min(pri.get(p, 99) for p in x.split("|")))errors_df = errors_df.sort_values("priority").reset_index(drop=True)errors_df.to_csv("earnings_errors_for_augmentation.csv", index=False)print(f"Exported: {len(errors_df)} errors to earnings_errors_for_augmentation.csv")print(f"\nBy confusion: "); print(errors_df.groupby(["true_label_name","pred_label_name"]).size())print(f"\nBy pattern: "); print(errors_df["linguistic_pattern"].value_counts())print(f"\n{'='*60}")print("NEXT STEPS:")print("1. Download earnings_errors_for_augmentation.csv")print("2. In Claude Code, run the verbalized-sampling-augment skill:")print("   'Augment the errors in data/earnings_errors_for_augmentation.csv'")print("3. Upload the generated CSV to Kaggle and run NB17b")

In [ ]:
# Show samplesfor _, r in errors_df.head(8).iterrows():    print(f"True={r['true_label_name']}, Pred={r['pred_label_name']} [{r['linguistic_pattern']}]")    print(f"  {r['text'][:120]}...\n")

## 9. Summary

In [ ]:
print(f"{'Metric':<25} {'Baseline':>10} {'Extended':>10} {'Delta':>10}")print("-"*58)for k in ["fpb_50_acc","fpb_all_acc","earnings_acc","overall_acc","source_3_acc","source_8_acc"]:    b, a = baseline_results.get(k), results_a.get(k)    if b and a:        print(f"  {k:<23} {b:>10.4f} {a:>10.4f} {a-b:>+10.4f}")del baseline_model; gc.collect(); torch.cuda.empty_cache()